In [55]:
import pandas as pd
import numpy as np
from pathlib import Path

TASK_ROOT = Path.cwd().parent
INPUT = TASK_ROOT/'input'
OUTPUT = TASK_ROOT/'output'

In [56]:
df_traj_all = pd.read_csv(INPUT/'all-trajectories.csv')
df_traj_full = df_traj_all.loc[
    df_traj_all.groupby("dblp_id")["CareerAgeZero"].transform("max").eq(20)
].copy()


In [57]:
stage_spans = [
    (0, 0,  "0.70", "0"),
    (1, 4,  "0.78", "1-4"),
    (5, 7,  "0.86", "5-7"),
    (8, 20, "0.93", "8-20")
]

EPS = 0.49

In [58]:
df_traj_full["log_pubs_adj"] = np.log(df_traj_full["pubs_adj"] + EPS)
df_traj_full["log_pubs_next"] = df_traj_full.groupby("dblp_id")["log_pubs_adj"].shift(-1)
df_traj_full["log_delta"] = df_traj_full["log_pubs_next"] - df_traj_full["log_pubs_adj"]
df_traj_full["state"] = df_traj_full["pubs_adj"].gt(0).astype(int)
df_traj_full["next_state"] = df_traj_full["pubs_adj_next"].gt(0).astype(int)


stage_bins = [-1, 0, 4, 7, 20]
stage_labels = [span[3] for span in stage_spans]

df_traj_full["stage"] = pd.cut(
    df_traj_full["CareerAgeZero"],
    bins=stage_bins,
    labels=stage_labels
)


In [59]:
year_logprod_rows = []

obs_years = np.arange(20 + 1)
transition_years = np.arange(20)

for year in obs_years:
    subset = df_traj_full[df_traj_full["CareerAge"] == year]
    vals = subset["log_pubs_adj"].dropna()

    year_logprod_rows.append({
        "year": year,
        "n": len(vals),
        "mean_log_productivity": vals.mean(),
        "var_log_productivity": vals.var(ddof=0),
        "sd_log_productivity": vals.std(ddof=0)})

year_logprod_params = pd.DataFrame(year_logprod_rows)



In [60]:
stage_logprod_rows = []

stage_order = ["0", "1-4", "5-7", "8-20"]

for stage in stage_order:
    subset = df_traj_full[df_traj_full["stage"] == stage]
    vals = subset["log_pubs_adj"].dropna()

    stage_logprod_rows.append({
        "stage": stage,
        "n": len(vals),
        "mean_log_productivity": vals.mean(),
        "var_log_productivity": vals.var(ddof=0),
        "sd_log_productivity": vals.std(ddof=0)
    })

stage_logprod_params = pd.DataFrame(stage_logprod_rows)

In [61]:
panel = df_traj_full.pivot(index="dblp_id",
                 columns="CareerAge",
                 values="pubs_adj")

# Ensure columns are sorted numerically
panel = panel.sort_index(axis=1)

# Option A: fill missing years with 0 (typical for productivity counts)
panel_filled = panel.fillna(0.0)

# Convert to numpy
trajectory_empirical_np = panel_filled.to_numpy().T

trajectory_empirical_np.shape

(21, 591)

In [62]:
df_rolling_pubs_cs = (
    df_traj_all.sort_values(["dblp", "CareerAgeZero"])
      .assign(
          RollingPubs=lambda d:
              d.groupby("dblp")["pubs_adj"]
               .rolling(4, min_periods=4).sum()
               .reset_index(level=0, drop=True)
      )
      .dropna(subset=["RollingPubs"])
      .groupby("dblp", as_index=False)
      .agg(
          FourYearProd=("RollingPubs", "last"),
          TotalYears=("CareerAgeZero", "max")))

In [63]:
df_traj_all.to_csv(OUTPUT / "df_traj_all.csv", index=False)
df_rolling_pubs_cs.to_csv(OUTPUT/'df_rolling_pubs_cs.csv')
year_logprod_params.to_csv(OUTPUT/'year_logprod_params.csv')
stage_logprod_params.to_csv(OUTPUT/'stage_logprod_params.csv')